# Full-188 protocol, one stage at a time
Select a Colab GPU runtime. Run each cell in order and inspect its output before continuing. Put `GITHUB_TOKEN` in Colab Secrets. All experiment outputs are under `/content/drive/MyDrive/DAIC_WOZ/experiments/full188_seed42/`. The original AVEC TEST roster is redistributed, so the held-out result is internal and cannot be described as an official AVEC test score. Do not run the final test cell until the method and mode are frozen on validation.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import base64, os, subprocess, sys

drive.mount('/content/drive')
DAIC = Path('/content/drive/MyDrive/DAIC_WOZ')
assert DAIC.is_dir(), f'Missing {DAIC}'
REPO = 'engrkhubabahmad/reliability-aware-depression-kd'
BRANCH = 'feature/full188-fresh-teachers-student-seed42'
ROOT = Path('/content/reliability-aware-depression-kd')
token = userdata.get('GITHUB_TOKEN')
assert token, 'Add GITHUB_TOKEN to Colab Secrets'
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
auth_env = os.environ.copy()
auth_env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.extraHeader',
                GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {auth}')
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', f'https://github.com/{REPO}.git', str(ROOT)],
                   env=auth_env, check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH],
               env=auth_env, check=True)
revision = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
CODE = Path('/content/reliability-aware-depression-kd-full188-' + revision[:12])
if not CODE.exists():
    subprocess.run(['git', '-C', str(ROOT), 'worktree', 'add', '--detach', str(CODE), revision], check=True)
else:
    assert (CODE / '.git').exists()
    assert subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip() == revision
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements-colab.txt')])
del token, auth, auth_env
print('Code revision:', revision)


## 1. Load the current checkout
The setup cell above fetches a fresh worktree for each commit. This cell removes stale Python imports from earlier attempts, verifies the text-teacher fix, and defines the stage runner. Run it after every checkout change.


In [ ]:
import json, pandas as pd, numpy as np
assert 'author.USE_PAGERANK = False' in (CODE / 'scripts/experiments/full188/train_text_teacher.py').read_text()
for name in list(sys.modules):
    if name == 'scripts' or name.startswith('scripts.'):
        del sys.modules[name]
sys.path[:] = [str(CODE)] + [p for p in sys.path
    if not str(p).startswith('/content/reliability-aware-depression-kd-full188')]
from scripts.experiments.full188.run_all import main as run_stage
EXP = DAIC / 'experiments/full188_seed42'
print('Revision:', revision, '\nExperiment:', EXP)


## 2. Preprocessing and split audit
Verify metadata, labels, and the participant-disjoint 132/28/28 split. This stage does not train or extract audio. Participant 440 is excluded. Text vocabulary and audio normalization are learned later from TRAIN-132 only.


In [ ]:
run_stage(['--daic-root', str(DAIC), '--stage', 'split', '--seed', '42'])
manifest = pd.read_csv(EXP / 'split/manifest.csv')
assert len(manifest) == 188 and manifest.participant_id.nunique() == 188
print(pd.crosstab(manifest['split'], manifest['label']))
print('Sources:', manifest.groupby(['split', 'source_split']).size().to_dict())


## 3. Audio feature extraction
Reuse the existing 141 ComParE16 arrays and extract the 47 missing recordings. If already complete, this stage skips extraction. The extraction cache stays on Drive and is never loaded in full into RAM.


In [ ]:
run_stage(['--daic-root', str(DAIC), '--stage', 'features', '--seed', '42'])
print(json.loads((EXP / 'features/provenance.json').read_text()))


## 4. Verify the feature cache
Confirm the manifest covers the same 188 participants, split assignments and labels. Check each array through memory mapping. This step reads no student TEST labels to select a model.


In [ ]:
cache = pd.read_csv(EXP / 'features/participant_manifest.csv').sort_values('participant_id')
split = manifest.sort_values('participant_id')
assert cache.participant_id.tolist() == split.participant_id.tolist()
assert cache.label.tolist() == split.label.tolist()
assert cache.split.tolist() == split.split.tolist()
for row in cache.itertuples(index=False):
    x = np.load(row.feature_path, mmap_mode='r', allow_pickle=False)
    assert x.ndim == 2 and x.shape[0] == 130 and x.shape[1] > 0
print('Verified arrays:', len(cache))
print(cache.groupby(['split', 'provenance']).size().to_string())


## 5. Train and evaluate the text teacher
This fresh Idiap InducT-GCN uses TRAIN-132 for fitting and VAL-28 for model selection. Previous incomplete attempts are preserved under `failed_attempts/`. Inspect the validation confusion matrix and report before proceeding.


In [ ]:
run_stage(['--daic-root', str(DAIC), '--stage', 'train_text_teacher', '--archive-incomplete', '--seed', '42'])
t = json.loads((EXP / 'teachers/text/audit.json').read_text())
print('Best epoch:', t['best_epoch'], 'VAL:', {k:v for k,v in t['val'].items() if k not in ('confusion_matrix','classification_report')})
print('VAL CM [0,1]:', t['val']['confusion_matrix'])
display(pd.DataFrame(t['val']['classification_report']).T)


## 6. Train and evaluate the audio teacher
The fresh USSD ComParE16 model fits audio normalization on TRAIN-132 only. Inspect the VAL-28 scores before training students.


In [ ]:
run_stage(['--daic-root', str(DAIC), '--stage', 'train_audio_teacher', '--archive-incomplete', '--seed', '42'])
a = json.loads((EXP / 'teachers/audio/audit.json').read_text())
print('Best epoch:', a['best_epoch'], 'VAL:', {k:v for k,v in a['val'].items() if k not in ('confusion_matrix','classification_report')})
print('VAL CM [0,1]:', a['val']['confusion_matrix'])
display(pd.DataFrame(a['val']['classification_report']).T)


## 7. Confirm TRAIN teacher targets
Only after both teacher audits pass, confirm each TRAIN-132 target file has the exact split IDs and labels. These are in-sample targets; no OOF run is used.


In [ ]:
train = manifest.loc[manifest.split.eq('train')].sort_values('participant_id')
for modality in ('text', 'audio'):
    path = EXP / f'teachers/{modality}/train_{modality}_targets.csv'
    target = pd.read_csv(path).sort_values('participant_id')
    assert target.participant_id.tolist() == train.participant_id.tolist()
    assert target.label.tolist() == train.label.tolist()
    assert target[f'{modality}_probability'].between(0, 1).all()
    print(modality, 'TRAIN targets verified:', len(target))


## 8. Train independent student text and audio branches


In [ ]:
run_stage(['--daic-root', str(DAIC), '--stage', 'train_student_branches', '--archive-incomplete', '--seed', '42'])
print((EXP / 'students/branches/audit.json').read_text())


## 9. Train and compare fusion modes on VAL
Fit `no_kd`, `standard_kd` and `ra_kd` with matched settings. Examine validation metrics and missing/noisy modality behavior. Do not open student TEST while selecting the method.


In [ ]:
run_stage(['--daic-root', str(DAIC), '--stage', 'train_student_fusion', '--archive-incomplete', '--seed', '42'])
f = json.loads((EXP / 'students/fusion/audit.json').read_text())
display(pd.DataFrame({mode: value['val'] for mode, value in f['results'].items()}).T)
robustness = pd.read_csv(EXP / 'students/fusion/val_missing_noise.csv')
display(robustness.groupby(['mode','scenario','noise_sd'])[['macro_f1','depressed_f1','auroc']].agg(['mean','std']).round(3))


## 10. Final student test: run once after freezing one mode
Run only after reviewing VAL and choosing one mode. This is the **internal** held-out 28-person split, not the official AVEC test partition. The scorer writes a selection lock.


In [ ]:
CHOSEN_MODE = 'ra_kd'  # Set once from VAL: no_kd, standard_kd, or ra_kd
subprocess.run([sys.executable, '-m', 'scripts.experiments.full188.score_student_test',
                '--daic-root', str(DAIC), '--split-dir', str(EXP / 'split'),
                '--features', str(EXP / 'features'),
                '--student-branches', str(EXP / 'students/branches'),
                '--audio-teacher', str(EXP / 'teachers/audio'),
                '--student-fusion', str(EXP / 'students/fusion'),
                '--mode', CHOSEN_MODE, '--output', str(EXP / 'students/internal_test')],
               cwd=CODE, check=True)
print((EXP / 'students/internal_test/result.json').read_text())
